# Federated experiments A to D on Colab

Runs the four experiments of the project and produces the real results for the web app's **FL experiments** and **Results & analytics** pages.

| | Client selection | Compression |
|---|---|---|
| A | random | none (baseline) |
| B | adaptive | none |
| C | random | Top-k (keeps 10%) |
| D | adaptive | Top-k (proposed method) |

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`.
2. On your PC run `python colab\make_zip.py` inside the `ml` folder so `ml/ml_for_colab.zip` is current (it must contain the latest `compare.py`).
3. Keep the tab open and the PC awake. Free Colab resets idle sessions and everything in `/content` is lost, so **run the backup cell after each experiment**.

**Rules that keep the results honest**
- Do not change any setting (`configs/base.yaml`, selection weights, Top-k ratio) after you have seen results. Decide first, then run.
- The test set is scored once, at the end of each run. Do not re-run an experiment just to get a nicer number.
- Report all four experiments, including ones that did not help.
- The smoke test below only proves the code runs. Its numbers mean nothing.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Upload the code and install

In [ ]:
%cd /content
!rm -rf /content/ml /content/ml_for_colab*.zip
from google.colab import files
up = files.upload()   # choose ml/ml_for_colab.zip (the .zip, not a .py file)
zips = [n for n in up if n.endswith(".zip")]
assert zips, "You must choose the .zip file, not a .py file"
!mkdir /content/ml
!unzip -q -o "/content/{zips[0]}" -d /content/ml
%cd /content/ml
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "flwr[simulation]==1.42.0"
%env HF_HUB_DISABLE_PROGRESS_BARS=1
import flwr, ray, torch
FLAGS = ""   # becomes "--cpu" if the GPU crashes (see section 3b)
print("flwr", flwr.__version__, "| ray", ray.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

Yellow dependency warnings are fine. Stop only for a red error, and paste it back.

## 3. Smoke test (gate: do not continue if this fails)

Two tiny rounds of experiment D. It checks that Flower, Ray and the GPU work together on this Colab. **Its accuracy is meaningless.** It should end with `Saved ... result.json`.

In [ ]:
!python -m src.federated.run_experiment --exp D --smoke $FLAGS

### 3b. Only if the smoke test crashed with `SIGFPE` / `dropout_cuda` / `ActorDiedError`

That is a GPU crash inside the Ray worker processes (seen on a fresh Colab). First this quick test shows whether the GPU also fails **outside** Ray. Then the fallback runs everything on the CPU, which is slower but avoids the crash.

In [ ]:
import torch
from src.models.mobilenet import build_model
m = build_model(16, pretrained=False).cuda().train()
out = m(torch.randn(8, 3, 96, 96).cuda())
out.sum().backward()
print("GPU training step worked outside Ray:", tuple(out.shape))

In [ ]:
# Fallback: every later cell uses $FLAGS, so they all run on the CPU from now on.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""   # hide the GPU from every process started from here
FLAGS = "--cpu"
!python -m src.federated.run_experiment --exp D --smoke $FLAGS

Tell me which of the two cells above failed: if the first one crashed, the GPU build on this Colab is broken; if only the Ray run crashed, it is a Ray/GPU-sharing problem. On CPU each experiment takes longer (Colab has only about 2 cores), so note the time of experiment A before starting the others.

## 4. The real experiments

Each cell is one experiment (20 rounds, 8 clients, 4 per round, uneven data split, seed 42). Run them **in order**. Note how long **A** takes: that tells you what to expect for the rest, and for extra seeds in section 7. The first run also downloads and prepares the dataset, so it is slower than the others.

In [ ]:
import time; t0 = time.time()
!python -m src.federated.run_experiment --exp A $FLAGS
print(f"Experiment A took {(time.time()-t0)/60:.1f} minutes")

In [ ]:
import time; t0 = time.time()
!python -m src.federated.run_experiment --exp B $FLAGS
print(f"Experiment B took {(time.time()-t0)/60:.1f} minutes")

In [ ]:
import time; t0 = time.time()
!python -m src.federated.run_experiment --exp C $FLAGS
print(f"Experiment C took {(time.time()-t0)/60:.1f} minutes")

In [ ]:
import time; t0 = time.time()
!python -m src.federated.run_experiment --exp D $FLAGS
print(f"Experiment D took {(time.time()-t0)/60:.1f} minutes")

## 5. Backup to Google Drive (optional but recommended)

Run this after **each** experiment. If Colab resets, the finished runs are safe in Drive (`MyDrive/tobacco_fl_outputs`). It asks you to allow Drive access.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import glob, os, shutil
dst = "/content/drive/MyDrive/tobacco_fl_outputs"
os.makedirs(dst, exist_ok=True)
for d in glob.glob("/content/ml/outputs/*-full-s*"):
    shutil.copytree(d, os.path.join(dst, os.path.basename(d)), dirs_exist_ok=True)
print("In Drive:", sorted(os.listdir(dst)))

## 6. Compare and download

Run once all four are finished. The comparison table shows test accuracy, macro-F1, rounds to converge and communication cost, **relative to A**.

In [ ]:
!python -m src.export.compare --seed 42

In [ ]:
import shutil, glob
from google.colab import files
shutil.make_archive("fl_results", "zip", "outputs", None)  # whole outputs folder
files.download("fl_results.zip")
print("Runs included:", sorted(g.split("/")[-1] for g in glob.glob("outputs/*-full-s*")))

Unzip `fl_results.zip` into `ml/outputs/` on your PC. Folders named `A-full-s42`, `B-full-s42`, ... must appear there.
Smoke folders (`*-smoke-*`) in the zip are not real results and are ignored by the comparison and import unless you ask for them.

## 7. Extra seeds (optional, for a paper)

One run per experiment can be luck. Repeating with other seeds shows whether differences are real. **Cost: each seed repeats all four experiments** (use the time you noted for A to estimate it). Run this only after section 6 has worked. Seeds 7 and 123 are fixed in advance on purpose.

In [ ]:
for seed in [7, 123]:
    for exp in "ABCD":
        print(f"\n=== seed {seed}, experiment {exp} ===", flush=True)
        !python -m src.federated.run_experiment --exp {exp} --seed {seed} $FLAGS

In [ ]:
import glob, json, statistics as st
from collections import defaultdict
runs = defaultdict(list)
for p in glob.glob("outputs/*-full-s*/result.json"):
    r = json.load(open(p)); runs[r["id"][0]].append(r)
def ms(v): return f"{st.mean(v):.3f} +/- {st.stdev(v):.3f}" if len(v) > 1 else f"{v[0]:.3f} (1 seed)"
print(f"{'Exp':4}{'seeds':>6}  {'test accuracy':24}{'macro-F1':24}{'total MB':>12}")
for k in sorted(runs):
    rs = runs[k]
    print(f"{k:4}{len(rs):>6}  {ms([r['summary']['accuracy'] for r in rs]):24}{ms([r['summary']['macroF1'] for r in rs]):24}"
          f"{st.mean(r['summary']['totalCommBytes'] for r in rs)/1e6:>12.1f}")

Read the table honestly: if the +/- ranges of two experiments overlap, you cannot claim one is better. That is a valid finding too.